# EconAI — Predictive Market Analytics

**Demand forecasting for Tier-2 city markets** using time-series + multivariable regression — Python, Pandas, Scikit-learn, with Power BI for dashboarding.

> **About this notebook.** This is a clean, fully runnable reconstruction of the EconAI methodology using **synthetic data**, so it reproduces anywhere with no external files. Replace the synthetic dataset in Section 2 with your real data (same column names) to regenerate every result on actual numbers.

**Author:** Kalash Jain · kalashj727@gmail.com

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)

def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

## 2. Build a synthetic Tier-2 demand dataset

Monthly demand driven by trend, seasonality, price, promotions, festival months (Oct/Nov), and a local income index, plus noise.

In [ ]:
n_months = 60
dates = pd.date_range('2020-01-01', periods=n_months, freq='MS')
t = np.arange(n_months)

trend = 1000 + 8 * t
season = 120 * np.sin(2 * np.pi * (t % 12) / 12)
price = 50 + 5 * np.random.randn(n_months)
promo = np.random.binomial(1, 0.30, n_months)
festival = ((dates.month == 10) | (dates.month == 11)).astype(int)
income_index = 100 + 0.5 * t + 3 * np.random.randn(n_months)
noise = 60 * np.random.randn(n_months)

demand = (trend + season - 9 * (price - 50) + 150 * promo
          + 200 * festival + 4 * (income_index - 100) + noise)
demand = np.clip(demand, 0, None)

df = pd.DataFrame({
    'demand': demand, 'price': price, 'promo': promo,
    'festival': festival, 'income_index': income_index
}, index=dates)
df.index.name = 'date'
df.head()

## 3. Exploratory data analysis

In [ ]:
print(df.describe().round(1))

ax = df['demand'].plot(title='Monthly demand — Tier-2 city (synthetic)')
ax.set_ylabel('Units')
plt.tight_layout()
plt.show()

## 4. Feature engineering

Calendar (`month`, `time_idx`), lagged demand (`lag_1`, `lag_2`, `lag_12`), and a 3-month rolling mean.

In [ ]:
d = df.copy()
d['time_idx'] = np.arange(len(d))
d['month'] = d.index.month
for lag in [1, 2, 12]:
    d['lag_' + str(lag)] = d['demand'].shift(lag)
d['roll_3'] = d['demand'].shift(1).rolling(3).mean()
d = pd.get_dummies(d, columns=['month'], drop_first=True)
d = d.dropna()
print('Rows after feature engineering:', len(d))
d.head()

## 5. Time-based train / test split

We hold out the **last 12 months** so the test set is strictly in the future (no leakage).

In [ ]:
test_size = 12
features = [c for c in d.columns if c != 'demand']
train, test = d.iloc[:-test_size], d.iloc[-test_size:]
X_train, y_train = train[features], train['demand']
X_test, y_test = test[features], test['demand']
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 6. Baseline: seasonal-naive forecast

A fair baseline = demand from the same month last year. Any real model must beat this.

In [ ]:
base_pred = test['lag_12'].values
base_mape = mape(y_test, base_pred)
base_rmse = rmse(y_test, base_pred)
print('Seasonal-naive baseline -> MAPE: %.2f%%  RMSE: %.1f' % (base_mape, base_rmse))

## 7. Models: Linear Regression and Random Forest

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr_mape, lr_rmse = mape(y_test, lr_pred), rmse(y_test, lr_pred)
rf_mape, rf_rmse = mape(y_test, rf_pred), rmse(y_test, rf_pred)
print('Linear Regression -> MAPE: %.2f%%  RMSE: %.1f' % (lr_mape, lr_rmse))
print('Random Forest     -> MAPE: %.2f%%  RMSE: %.1f' % (rf_mape, rf_rmse))

## 8. Results and error comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Seasonal naive (baseline)', 'Linear Regression', 'Random Forest'],
    'MAPE_%': [base_mape, lr_mape, rf_mape],
    'RMSE': [base_rmse, lr_rmse, rf_rmse]
}).round(2)

best_mape = min(lr_mape, rf_mape)
improvement = (base_mape - best_mape) / base_mape * 100
print(results.to_string(index=False))
print('Best model reduces MAPE by %.1f%% vs the seasonal-naive baseline.' % improvement)

## 9. Actual vs forecast

In [ ]:
if rf_mape <= lr_mape:
    best_name, best_pred = 'Random Forest', rf_pred
else:
    best_name, best_pred = 'Linear Regression', lr_pred

plt.plot(y_test.index, y_test.values, label='Actual', marker='o')
plt.plot(y_test.index, best_pred, label=best_name + ' forecast', marker='o')
plt.title('Actual vs forecast — 12-month hold-out')
plt.ylabel('Units')
plt.legend()
plt.tight_layout()
plt.show()

## 10. What drives demand?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).head(10)
importances[::-1].plot(kind='barh', title='Top demand drivers (Random Forest importance)')
plt.tight_layout()
plt.show()

## 11. Use this on your real data

1. Replace Section 2 with `df = pd.read_csv('your_data.csv', parse_dates=['date'], index_col='date')`.
2. Keep columns: `demand`, plus any drivers (price, promo, festival, income_index, ...).
3. Re-run all cells. The metrics, charts, and driver ranking update automatically.
4. Export predictions to CSV and load them into Power BI for the dashboard view.